# 본인 Supabase Postgres · 원본 SQL 5개 적재

본인 Supabase 프로젝트를 생성하고 Connect → Session pooler의 정보를 확인하세요. 파일 → Drive에 사본 저장 후 실행합니다. 이미 적재했으면 생성·INSERT 셀은 건너뜁니다.

[전체 안내](https://bdai13-data-handling.vercel.app/ch05#load-all-tables)

In [ ]:
!pip install psycopg2-binary -q

In [ ]:
from getpass import getpass
import psycopg2
import pandas as pd

# 본인 프로젝트의 Connect → Session pooler에서 각각 복사하세요.
conn = psycopg2.connect(
    host=input("본인 Session pooler Host: ").strip(),
    port=int(input("Port [5432]: ").strip() or "5432"),
    dbname="postgres",
    user=input("User (postgres.프로젝트참조값): ").strip(),
    password=getpass("본인 프로젝트의 Database password: "),
    sslmode="require",
    connect_timeout=20,
)
conn.autocommit = True

def query(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[col[0] for col in cur.description])

query("SELECT current_database(), current_user")

## 테이블 생성 + 원본 SQL 5개 적재
본인의 빈 bdai13 스키마에서 한 번만 실행합니다. CREATE 실패·INSERT 실패는 전체 rollback됩니다. 비밀번호를 코드에 적지 마세요.

In [ ]:
import requests

base = "https://raw.githubusercontent.com/team-datapopcorn/bdai13-data-handling-weekly/master/"
paths = ["week05_sql_aggregation/00_create_tables_postgres.sql"] + [
    f"insert_{t}.sql" for t in ["customers", "products", "orders", "order_items", "logins"]
]
scripts = []
for path in paths:
    r = requests.get(base + path, timeout=30)
    r.raise_for_status()
    scripts.append(r.text)

# 생성문 파일의 BEGIN/COMMIT은 제거하고, 전체를 하나의 트랜잭션으로 실행합니다.
scripts[0] = scripts[0].removeprefix("BEGIN;\n").removesuffix("COMMIT;\n")
conn.autocommit = False
try:
    with conn.cursor() as cur:
        for script in scripts:
            cur.execute(script)
    conn.commit()
    print("테이블 5개 생성 및 전체 적재 완료")
except Exception:
    conn.rollback()
    raise
finally:
    conn.autocommit = True

## 건수 확인
customers 400 / products 10 / orders 713 / order_items 1782 / logins 1953

In [ ]:
query("""
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM bdai13.customers
UNION ALL
SELECT 'products' AS table_name, COUNT(*) AS row_count FROM bdai13.products
UNION ALL
SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM bdai13.orders
UNION ALL
SELECT 'order_items' AS table_name, COUNT(*) AS row_count FROM bdai13.order_items
UNION ALL
SELECT 'logins' AS table_name, COUNT(*) AS row_count FROM bdai13.logins;
""")

In [ ]:
query("SELECT * FROM bdai13.orders ORDER BY order_id LIMIT 5")

## 같은 DB에서 JOIN·집계

In [ ]:
query("""SELECT c.region, COUNT(*) AS order_count, SUM(o.order_total) AS total_sales
FROM bdai13.orders o JOIN bdai13.customers c ON c.customer_id = o.customer_id
GROUP BY c.region ORDER BY c.region""")

## 수업 종료

In [ ]:
conn.close()